# Chapter 1 — Claude API Fundamentals · Practice

**Goal:** take one `POST /v1/messages` call apart and rebuild it as a multi-turn chat helper that handles every `stop_reason` and reports token usage and cost.

- **Theory:** [README.md](README.md) (read it first; this notebook only recaps)
- **Estimated time:** 2.5–3.5 hours
- **Estimated API cost:** about $0.15 with the default model (`claude-sonnet-5-5`, low effort). Optional extras are marked and add a few cents. Offline cells are free.
- **Prerequisites:** [00 — Prerequisites](../../00-prerequisites/README.md) done: Python 3.11+, the repo venv with `requirements.txt` installed, `.env` with `ANTHROPIC_API_KEY`.

**How to read the cells.** Code comments start with `# [live]` when a cell calls the API (costs money, needs a key) and `# [offline]` when it uses mock responses or local calculations (free, works without a key). Run the notebook top to bottom once; then come back to the **Try it** prompts.

## 0. Setup
Imports, `.env` loading and the default model. Override the model in `.env` with `CLAUDE_MODEL=...` instead of editing code.

In [ ]:
import hashlib
import json
import os
import re
import statistics
import time
from dataclasses import dataclass, field
from types import SimpleNamespace
from typing import Any, Callable

import anthropic
from anthropic.types import Message
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())                       # reads ANTHROPIC_API_KEY (and optional CLAUDE_MODEL) from .env
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
HAIKU = "claude-haiku-4-5-20251001"              # cheapest model; retirement "not sooner than Oct 15, 2026"
client = anthropic.Anthropic()                   # creating the client makes no network call

print("anthropic SDK", anthropic.__version__, "| default model:", MODEL)

In [ ]:
# Live cells need a key. Offline cells (mocks, graders, calculators) work without one.
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see ../../00-prerequisites/README.md)"

Small helpers used throughout: `cheap()` keeps live calls inexpensive (`max_tokens=2048`, `effort="low"`), `text_of()` reads text safely, `show()` prints what matters, and `fake_message()` builds realistic **offline** responses with the SDK's own `Message` type.

In [ ]:
def supports_effort(model: str) -> bool:
    """Haiku 4.5 has no `effort` setting; the 5.x models do."""
    return not model.startswith("claude-haiku")


def cheap(model: str = MODEL, effort: str = "low", max_tokens: int = 2048) -> dict[str, Any]:
    """Request options for low-cost demo calls: modest output budget, low effort where supported."""
    opts: dict[str, Any] = {"model": model, "max_tokens": max_tokens}
    if supports_effort(model):
        opts["output_config"] = {"effort": effort}
    return opts


def text_of(resp) -> str:
    """Join the text blocks of a response. Never use resp.content[0].text."""
    return "".join(b.text for b in resp.content if b.type == "text")


def show(resp) -> None:
    """Print only the parts of a response that matter."""
    u = resp.usage
    print("id:          ", resp.id)
    print("model:       ", resp.model)
    print("stop_reason: ", resp.stop_reason, "| stop_sequence:", repr(resp.stop_sequence))
    print("blocks:      ", [b.type for b in resp.content])
    print(f"usage:        input={u.input_tokens} output={u.output_tokens} "
          f"cache_read={u.cache_read_input_tokens or 0} cache_write={u.cache_creation_input_tokens or 0}")
    print("text:        ", text_of(resp)[:600])


def fake_message(content: list[dict] | None = None, stop_reason: str = "end_turn", *, model: str = MODEL,
                 stop_sequence: str | None = None, stop_details: dict | None = None,
                 input_tokens: int = 100, output_tokens: int = 20) -> Message:
    """Build a real anthropic.types.Message from plain dicts, for offline demos and tests."""
    return Message.model_validate({
        "id": "msg_offline_demo", "type": "message", "role": "assistant", "model": model,
        "content": [{"type": "text", "text": "Hello!"}] if content is None else content,
        "stop_reason": stop_reason, "stop_sequence": stop_sequence, "stop_details": stop_details,
        "usage": {"input_tokens": input_tokens, "output_tokens": output_tokens},
    })


print("helpers ready")

## 1.1 Anatomy of a Messages API request

Every interaction with Claude is one endpoint, `POST /v1/messages`. Three fields are required (`model`, `max_tokens`, `messages`); everything else (`system`, `tools`, `stop_sequences`, `output_config`, `thinking`, `metadata`, …) is an optional parameter on the same endpoint. The response is a `Message` whose `content` is a **list of typed blocks**, plus `stop_reason` and `usage`.

[Theory](README.md#11-anatomy-of-a-messages-api-request)

In [ ]:
# [offline] The request is just JSON. This is exactly what the SDK sends.
request = {
    "model": MODEL,                                                   # required
    "max_tokens": 1024,                                               # required: a ceiling, thinking included
    "system": "You are a concise assistant for a cloud architecture team.",  # optional, top level
    "messages": [                                                     # required: the full conversation
        {"role": "user", "content": "In two sentences, what is an API gateway?"},
    ],
}
print("POST https://api.anthropic.com/v1/messages")
print("headers: x-api-key: <from .env> | anthropic-version: 2023-06-01 | content-type: application/json")
print(json.dumps(request, indent=2))

In [ ]:
# [live] ~ $0.003
response = client.messages.create(**request)
show(response)
print("\nRaw JSON body as returned by the API:")
print(response.model_dump_json(indent=2, exclude_none=True))

> **Observe:** `content` is a list. On Sonnet 5.5 adaptive thinking is on by default, so you may see a `thinking` block **before** the `text` block, with an empty `thinking` field (thinking text is omitted by default) and a `signature`. `usage` reports input and output tokens; thinking is billed as output.

Why `response.content[0].text` is a bug waiting to happen, shown offline:

In [ ]:
# [offline] A thinking block first, then text: the common shape on current models
thinking_first = fake_message([
    {"type": "thinking", "thinking": "", "signature": "sig_demo"},
    {"type": "text", "text": "An API gateway is a single entry point that routes, secures and meters API traffic."},
])
try:
    print(thinking_first.content[0].text)
except AttributeError as e:
    print("content[0].text ->", type(e).__name__ + ":", e)
print("text_of()       ->", text_of(thinking_first))

# A refusal arrives with HTTP 200 and EMPTY content: content[0] would be an IndexError
refused = fake_message([], "refusal", stop_details={"type": "refusal", "category": None, "explanation": None})
print("refusal content:", refused.content, "| text_of() ->", repr(text_of(refused)))

**Errors are not stop reasons.** A failed request raises a typed exception; a successful one returns HTTP 200 with a `stop_reason`. The SDK retries connection errors, 408, 409, 429 and 5xx (including 529 overloaded) twice by default with exponential backoff.

In [ ]:
# [offline] The SDK's exception classes and which ones the client retries automatically
STATUS = {
    anthropic.BadRequestError: 400, anthropic.AuthenticationError: 401, anthropic.PermissionDeniedError: 403,
    anthropic.NotFoundError: 404, anthropic.RequestTooLargeError: 413, anthropic.RateLimitError: 429,
    anthropic.InternalServerError: 500, anthropic.ServiceUnavailableError: 503, anthropic.OverloadedError: 529,
}
for cls, status in STATUS.items():
    retried = status in (408, 409, 429) or status >= 500
    print(f"{status}  {cls.__name__:<24} retried by SDK: {retried}")
print("--   APIConnectionError / APITimeoutError  retried by SDK: True")
print("default max_retries:", client.max_retries, "| change with anthropic.Anthropic(max_retries=..., timeout=...)")

In [ ]:
# [live] Trigger two errors on purpose. Failed requests like these are not billed.
def try_call(label: str, c: anthropic.Anthropic, **overrides) -> None:
    kwargs = {"model": MODEL, "max_tokens": 64, "messages": [{"role": "user", "content": "ping"}]} | overrides
    try:
        c.messages.create(**kwargs)
        print(f"{label}: succeeded")
    except anthropic.APIStatusError as e:          # any HTTP 4xx/5xx; subclasses carry the exact type
        print(f"{label}: {type(e).__name__} (HTTP {e.status_code}) request_id={e.request_id}")
    except anthropic.APIConnectionError as e:       # DNS, TLS, refused connection, timeout
        print(f"{label}: {type(e).__name__}: {e}")

try_call("bad model id", client, model="claude-does-not-exist")
try_call("bad api key ", anthropic.Anthropic(api_key="sk-ant-invalid", max_retries=0))

> **Observe:** you branch on the exception **type** (`NotFoundError`, `AuthenticationError`, `RateLimitError`), never on the message string. Log `request_id`; support asks for it.

> **Try it:** add `stop_sequences=["."]` to `request` and rerun the live cell. What are `stop_reason` and `stop_sequence` now? Then try `metadata={"user_id": hashlib.sha256(b"ana@example.com").hexdigest()}`: an opaque ID, never the raw email.

## 1.2 Message roles, alternation and stateless conversations

`messages` alternates `user` and `assistant` turns. Tool results travel as `tool_result` blocks **inside a `user` message** (there is no `tool` role), and consecutive same-role turns are merged into one. The API is **stateless**: no session, no server-side memory, so your app resends every earlier turn. On supported models a mid-conversation `role: "system"` message can add an operator instruction without touching the cached prefix.

[Theory](README.md#12-message-roles-alternation-and-stateless-conversations)

In [ ]:
# [offline] A conversation with every common block type, and what the API does with back-to-back user turns
conversation = [
    {"role": "user", "content": "Where is order A17?"},
    {"role": "assistant", "content": [
        {"type": "text", "text": "Let me check."},
        {"type": "tool_use", "id": "toolu_01", "name": "get_order", "input": {"order_id": "A17"}},
    ]},
    {"role": "user", "content": [   # tool results go in a USER turn, linked by tool_use_id
        {"type": "tool_result", "tool_use_id": "toolu_01", "content": "Order A17: shipped 2026-10-01"},
    ]},
    {"role": "assistant", "content": "Order A17 shipped on October 1."},
]
for m in conversation:
    blocks = [m["content"]] if isinstance(m["content"], str) else [b["type"] for b in m["content"]]
    print(f"{m['role']:<9} {blocks}")


def as_blocks(content) -> list[dict]:
    """'Hi' is shorthand for [{'type': 'text', 'text': 'Hi'}]."""
    return [{"type": "text", "text": content}] if isinstance(content, str) else list(content)


def merge_consecutive(messages: list[dict]) -> list[dict]:
    """What the API does with consecutive same-role turns: combine them into one turn."""
    merged: list[dict] = []
    for m in messages:
        if merged and merged[-1]["role"] == m["role"] and m["role"] in ("user", "assistant"):
            merged[-1] = {"role": m["role"], "content": as_blocks(merged[-1]["content"]) + as_blocks(m["content"])}
        else:
            merged.append(dict(m))
    return merged

two_users = [{"role": "user", "content": "I'm vegetarian."}, {"role": "user", "content": "Suggest a dinner."}]
print("\nbefore:", [m["role"] for m in two_users], "-> after:", json.dumps(merge_consecutive(two_users)))

A minimal stateless chat. With `remember=False` it sends only the latest message, which reproduces the classic "the bot forgot my preference" bug.

In [ ]:
class StatelessChat:
    """The app owns the history and resends all of it on every call."""

    def __init__(self, system: str, model: str = MODEL, remember: bool = True):
        self.system, self.model, self.remember = system, model, remember
        self.history: list[dict] = []

    def send(self, user_text: str) -> str:
        turn = {"role": "user", "content": user_text}
        messages = self.history + [turn] if self.remember else [turn]
        resp = client.messages.create(**cheap(self.model), system=self.system, messages=messages)
        if self.remember:
            # Append the FULL content list (thinking/tool blocks included), not just the text
            self.history += [turn, {"role": "assistant", "content": resp.content}]
        return text_of(resp)

In [ ]:
# [live] 4 short calls, ~ $0.01
SYSTEM_TRAVEL = "You are a friendly travel assistant. Keep answers under 60 words."
for remember in (True, False):
    bot = StatelessChat(SYSTEM_TRAVEL, remember=remember)
    bot.send("I love hiking and I'm vegetarian.")
    print(f"--- remember={remember}\n{bot.send('Suggest one weekend activity and one dinner idea near Almaty.')}\n")

> **Observe:** with `remember=False` the dinner idea is no longer guaranteed to be vegetarian and the activity may not be a hike. Nothing is wrong with the model or the context window: the app simply did not send turn 1.

**Mid-conversation `system` message** (Fable 5 / 5.1, Opus 4.8 / 5 / 5.5, Sonnet 5.5; a `400` on Haiku 4.5 and Sonnet 5). It must follow a `user` turn and be followed by an `assistant` turn or end the array.

In [ ]:
# [live] ~ $0.002. A backend event arrives mid-chat; the top-level system prompt (and its cache) is untouched.
booking_chat = [
    {"role": "user", "content": "Hi, I booked a hotel in Almaty last week."},
    {"role": "assistant", "content": "Thanks! How can I help with your booking?"},
    {"role": "user", "content": "Any update on my booking?"},
    {"role": "system", "content": "Backend event: booking #A17 was confirmed 2 minutes ago."},
]
try:
    resp = client.messages.create(**cheap(), system="You are a booking assistant. Be brief.", messages=booking_chat)
    print(text_of(resp))
except anthropic.BadRequestError as e:
    print("400: this model does not accept mid-conversation system messages ->", e.message)

In [ ]:
# [live] Free (a 400 is not billed). Final-turn assistant prefill is rejected on Sonnet/Opus 4.6+ and all 5.x models.
prefill = [
    {"role": "user", "content": "Give me a JSON object with keys city and country for Almaty."},
    {"role": "assistant", "content": "{"},
]
try:
    resp = client.messages.create(**cheap(), messages=prefill)
    print("Accepted (an older model such as Haiku 4.5):", text_of(resp))
except anthropic.BadRequestError as e:
    print("400 as expected on current models ->", e.message)

> **Observe:** prefill fails on the default model, so use structured outputs (`output_config.format`, Chapter 6) or clear format instructions instead.

> **Try it:** move the `system` message in `booking_chat` to index 0, or put another `user` turn after it. Read the 400 message. Then rerun the cell with `CLAUDE_MODEL=claude-haiku-4-5-20251001`.

## 1.3 `stop_reason`: why the model stopped

`stop_reason` is the only reliable control signal. There are **seven** current values: `end_turn`, `tool_use`, `max_tokens`, `stop_sequence`, `pause_turn`, `refusal`, `model_context_window_exceeded`. Three are easy to trigger for real; the other four are simulated below with mock responses. `stop_details` is populated only for `refusal`.

[Theory](README.md#13-stop_reason-why-the-model-stopped)

In [ ]:
# [live] 3 calls, ~ $0.005. Same prompt, three ways to stop.
count_prompt = [{"role": "user", "content": "Count from 1 to 15, one number per line, nothing else."}]
cases = {
    "end_turn": {},
    "max_tokens": {"max_tokens": 10},          # far too small on purpose
    "stop_sequence": {"stop_sequences": ["7"]},
}
for label, extra in cases.items():
    resp = client.messages.create(**(cheap() | extra), messages=count_prompt)
    print(f"{label:<14} stop_reason={resp.stop_reason!r:<16} stop_sequence={resp.stop_sequence!r:<5} "
          f"blocks={[b.type for b in resp.content]} text={text_of(resp)!r}")

> **Observe:** with `max_tokens=10` the output is cut mid-way, and on a thinking model the whole budget can go to the `thinking` block, leaving no text at all. With `stop_sequences=["7"]` the text ends at 6 and the matched string is **not** included; `resp.stop_sequence` tells you which one fired.

A reusable dispatcher that maps every value to the next action. It returns a decision instead of raising, so it is easy to test:

In [ ]:
@dataclass
class Outcome:
    action: str          # what the caller should do next
    text: str = ""
    detail: str = ""


def handle(resp) -> Outcome:
    match resp.stop_reason:
        case "end_turn":
            return Outcome("return", text_of(resp))
        case "tool_use":
            names = [b.name for b in resp.content if b.type == "tool_use"]
            return Outcome("run_tools", text_of(resp), f"run ALL of {names}, send every tool_result in one user turn")
        case "pause_turn":
            return Outcome("resend_unchanged", "", "append resp.content as the assistant turn and call again; no extra user message")
        case "max_tokens":
            return Outcome("raise_budget", text_of(resp), "truncated: raise max_tokens, stream or continue; never run a tool_use from this turn")
        case "model_context_window_exceeded":
            return Outcome("shrink_input", text_of(resp), "window full: trim, summarize, compact or split the input")
        case "stop_sequence":
            return Outcome("return", text_of(resp), f"stopped on {resp.stop_sequence!r}")
        case "refusal":
            category = resp.stop_details.category if resp.stop_details else None
            return Outcome("refused", "", f"category={category}: explain, or retry on a fallback model")
        case other:
            return Outcome("unknown", text_of(resp), f"unhandled stop_reason {other!r}: log it and fail safe")

In [ ]:
# [offline] One mock per stop_reason, shaped like the real responses in the docs
MOCKS = {
    "end_turn": fake_message(),
    "tool_use": fake_message([
        {"type": "text", "text": "Let me look that up."},
        {"type": "tool_use", "id": "toolu_01", "name": "get_order", "input": {"order_id": "A17"}},
    ], "tool_use"),
    "pause_turn": fake_message([
        {"type": "server_tool_use", "id": "srvtoolu_01", "name": "web_search", "input": {"query": "Claude API changes"}},
    ], "pause_turn"),
    "max_tokens": fake_message([{"type": "text", "text": "The three main reasons are: 1) lat"}], "max_tokens"),
    "model_context_window_exceeded": fake_message(
        [{"type": "text", "text": "Summary of filing 212: revenue grew"}], "model_context_window_exceeded"),
    "stop_sequence": fake_message([{"type": "text", "text": "1\n2\n3\n4\n5\n6\n"}], "stop_sequence", stop_sequence="7"),
    "refusal": fake_message([], "refusal", stop_details={
        "type": "refusal", "category": "cyber",
        "explanation": "This request was declined because it could enable cyber harm."}),
}
for name, mock in MOCKS.items():
    o = handle(mock)
    print(f"{name:<30} -> {o.action:<17} {o.detail}")
    assert (mock.stop_details is not None) == (name == "refusal")   # stop_details only on refusal

# A value this SDK version does not know yet (the beta endpoint can return "compaction"):
future = fake_message().model_copy(update={"stop_reason": "compaction"})
print(f"{'compaction (future value)':<30} -> {handle(future).action}")

In [ ]:
# [offline] Continuing after pause_turn: send the paused assistant turn back AS IS
user_turn = {"role": "user", "content": "Find this week's Claude API changes."}
paused = MOCKS["pause_turn"]
next_messages = [user_turn, {"role": "assistant", "content": paused.content}]   # no "Continue" user message
print("next request roles:", [m["role"] for m in next_messages])
print("last block sent back:", next_messages[-1]["content"][-1].type)

**Refusals and server-side fallback (beta).** On the Claude API you can ask the server to retry a declined request on a recommended fallback model in the same call. This harmless prompt should not be refused; the point is the request shape and where to look in the response.

In [ ]:
# [live] ~ $0.002. Beta: server-side fallback (Claude API only; not Bedrock, Google Cloud, Foundry or Batches)
try:
    resp = client.beta.messages.create(
        **cheap(),
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        messages=[{"role": "user", "content": "In one sentence, why should a web server hide its version banner?"}],
    )
    print("answered by:", resp.model, "| stop_reason:", resp.stop_reason, "| stop_details:", resp.stop_details)
    print("blocks:", [b.type for b in resp.content])     # a 'fallback' block would mark a model hand-off
except anthropic.BadRequestError as e:
    print("400 (model or account may not support this beta) ->", e.message)

> **Observe:** `stop_details` is `None` on a normal answer. On a refusal it is an object whose `category` and `explanation` may themselves be `None`, so branch on `stop_reason == "refusal"`, never on the category. If a fallback served the turn, `resp.model` names the fallback model.

> **Try it:** add a branch to `handle()` that treats `stop_sequence` differently when the matched sequence is `"</answer>"` (for example, parse the answer). Write a mock for it with `fake_message(..., "stop_sequence", stop_sequence="</answer>")`.

## 1.4 The system prompt

The top-level `system` field carries operator instructions for every turn (role, scope, rules, format), and your code resends it on every request. Wording matters: absolute or keyword-triggered rules ("ALWAYS call X", "if the user mentions *account*…") create unintended tool associations. In long chats, behavior can drift; short reminders at breakpoints help.

[Theory](README.md#14-the-system-prompt)

In [ ]:
# [offline] Two system prompts for the same support bot, plus a tiny "prompt linter"
ABSOLUTE_PROMPT = """You are the support assistant for Acme Cloud.
Always verify the customer first by calling get_customer.
Answer billing, plan and general product questions in at most 80 words."""

SCOPED_PROMPT = """You are the support assistant for Acme Cloud.
Answer billing, plan and general product questions in at most 80 words.
Before discussing a specific customer's account, order or invoice, look the customer up with get_customer.
General questions (support hours, plans, how-to steps) need no lookup; use search_faq for those."""

TOOLS = [
    {"name": "get_customer",
     "description": "Look up one customer's account record (plan, invoices, status) by email. "
                    "Use only for questions about that specific customer's account.",
     "input_schema": {"type": "object", "properties": {"email": {"type": "string"}}, "required": ["email"]}},
    {"name": "search_faq",
     "description": "Search Acme Cloud's public FAQ: support hours, plans, how-to guides.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
]


def lint_system_prompt(prompt: str, tool_names: list[str]) -> list[str]:
    """Flag wording that tends to over-trigger tools. A heuristic, not a guarantee."""
    warnings = []
    for n, line in enumerate(prompt.splitlines(), 1):
        if re.search(r"\b(always|every time|for every)\b", line, re.I) and any(t in line for t in tool_names):
            warnings.append(f"line {n}: absolute rule tied to a tool -> scope it to when it applies")
        if re.search(r"\bif the user (mentions|says|uses the word)\b", line, re.I):
            warnings.append(f"line {n}: keyword trigger -> describe the situation, not a word")
    return warnings

names = [t["name"] for t in TOOLS]
print("ABSOLUTE:", lint_system_prompt(ABSOLUTE_PROMPT, names))
print("SCOPED:  ", lint_system_prompt(SCOPED_PROMPT, names))

In [ ]:
# [live] 8 calls, ~ $0.02. A/B: which tool does the model ask for first under each prompt? (tools are NOT executed)
QUESTIONS = [
    "What are your support hours?",
    "How do I enable two-factor authentication?",
    "I'm ana@example.com. Why was I charged twice on invoice INV-88?",
    "Which plan includes SSO?",
]
print(f"{'question':<62} {'ABSOLUTE':<14} SCOPED")
for q in QUESTIONS:
    picks = []
    for system in (ABSOLUTE_PROMPT, SCOPED_PROMPT):
        resp = client.messages.create(**cheap(), system=system, tools=TOOLS,
                                      messages=[{"role": "user", "content": q}])
        calls = [b.name for b in resp.content if b.type == "tool_use"]
        picks.append(calls[0] if calls else "(none)")
    print(f"{q[:60]:<62} {picks[0]:<14} {picks[1]}")

> **Observe:** under `ABSOLUTE_PROMPT` the model tends to reach for `get_customer` even for "support hours", although the tool description says not to. The fix is the prompt wording, not the tool. Results vary run to run; look at the pattern.

**Drift experiment.** A strict format rule over several turns of chatty user messages, with and without a short reminder every second turn. On models that support it the reminder is a mid-conversation `system` message; otherwise it is prefixed to the user turn.

In [ ]:
# [live] 2 runs x 6 turns = 12 calls, ~ $0.03. Raise N_TURNS to see more drift (cost grows ~quadratically).
N_TURNS = 6
DRIFT_SYSTEM = "You are Acme's onboarding assistant. Reply in at most 30 words and end every reply with the tag [acme]."
REMINDER = "Reminder: at most 30 words, and end with [acme]."
MIDCONV_SYSTEM_MODELS = {"claude-fable-5-1", "claude-fable-5", "claude-opus-5-5", "claude-opus-5", "claude-opus-4-8", "claude-sonnet-5-5"}
CHATTY = [
    "Hi! I'm new. Can you walk me through absolutely everything about setting up my first project in detail?",
    "Great, now explain all the billing options with examples and edge cases, I love long answers.",
    "Tell me a long story about how a team migrated to Acme Cloud.",
    "Compare every plan in a detailed table please.",
    "Write me a step-by-step guide for backups with lots of detail.",
    "Summarize everything we discussed so far, as long as you like.",
]


def compliant(text: str) -> bool:
    return text.rstrip().endswith("[acme]") and len(text.split()) <= 31


def run_drift(remind_every: int | None) -> list[bool]:
    history: list[dict] = []
    marks = []
    for i in range(1, N_TURNS + 1):
        user_text = CHATTY[(i - 1) % len(CHATTY)]
        history.append({"role": "user", "content": user_text})
        if remind_every and i % remind_every == 0:
            if MODEL in MIDCONV_SYSTEM_MODELS:
                history.append({"role": "system", "content": REMINDER})
            else:
                history[-1] = {"role": "user", "content": f"({REMINDER})\n{user_text}"}
        resp = client.messages.create(**cheap(), system=DRIFT_SYSTEM, messages=history)
        history.append({"role": "assistant", "content": resp.content})
        marks.append(compliant(text_of(resp)))
    return marks

for label, every in (("no reminders  ", None), ("reminder every 2", 2)):
    marks = run_drift(every)
    print(f"{label}: {' '.join('✓' if m else '✗' for m in marks)}  ({sum(marks)}/{len(marks)} compliant)")

> **Observe:** current models often hold a short rule for a handful of turns; drift shows up in long sessions with long, pattern-setting replies. The cheapest fixes come first: concrete prompt, clearly tagged critical rules, then reminders. Validate-and-regenerate is the last resort.

> **Try it:** rewrite `ABSOLUTE_PROMPT` as `"If the user mentions account, call get_customer"` and rerun the linter and the A/B cell. Which questions change?

## 1.5 The context window

Everything in one request shares the window: system prompt, tool definitions, the whole `messages` history (tool results, documents, kept thinking blocks) and this turn's output. Sonnet 5.5, Opus 5.5 and Fable 5.1 have 1M tokens; Haiku 4.5 has 200K. More room is not better answers: curate what goes in (trim tool results, keep a "case facts" block, put key facts at the edges).

[Theory](README.md#15-the-context-window)

In [ ]:
# [offline] What happens for a given input size and max_tokens? (Claude 4.5+ behavior)
CONTEXT_WINDOW = {"claude-fable-5-1": 1_000_000, "claude-opus-5-5": 1_000_000, "claude-sonnet-5-5": 1_000_000, HAIKU: 200_000}
MAX_OUTPUT = {"claude-fable-5-1": 128_000, "claude-opus-5-5": 128_000, "claude-sonnet-5-5": 128_000, HAIKU: 64_000}


def predict_window_outcome(input_tokens: int, max_tokens: int, model: str = MODEL) -> str:
    if max_tokens > MAX_OUTPUT[model]:
        return "400: max_tokens is above the model's output limit"
    if input_tokens > CONTEXT_WINDOW[model]:
        return "400: prompt is too long (the input alone exceeds the window)"
    if input_tokens + max_tokens > CONTEXT_WINDOW[model]:
        return "accepted; generation may stop with model_context_window_exceeded"
    return "fits"

for inp, mx, m in [(600, 16_000, "claude-sonnet-5-5"), (950_000, 64_000, "claude-sonnet-5-5"),
                   (1_050_000, 1_000, "claude-sonnet-5-5"), (150_000, 60_000, HAIKU), (10_000, 100_000, HAIKU)]:
    print(f"{m:<26} input={inp:>9,} max_tokens={mx:>7,} -> {predict_window_outcome(inp, mx, m)}")

In [ ]:
# [offline] Tool-result bloat: a 40-field payload when the conversation needs 5 fields
raw_order = {f"field_{i:02d}": f"value {i} " * 6 for i in range(35)} | {
    "order_id": "A17", "status": "shipped", "carrier": "DHL", "eta": "2026-10-06", "total_usd": 42.50}
KEEP = ("order_id", "status", "carrier", "eta", "total_usd")
trimmed_order = {k: raw_order[k] for k in KEEP}
print(f"raw: {len(raw_order)} fields, {len(json.dumps(raw_order)):,} chars | trimmed: {len(trimmed_order)} fields, {len(json.dumps(trimmed_order))} chars")
print("Every later turn re-sends this tool_result, so the saving repeats on every call.")

In [ ]:
# [live] Free: the token counting endpoint does not bill tokens (it has its own rate limit)
for label, payload in (("raw", raw_order), ("trimmed", trimmed_order)):
    n = client.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content": json.dumps(payload)}]).input_tokens
    print(f"{label:<8} {n:>6,} input tokens")

**Hybrid history.** Keep exact values in a structured *case facts* block that is never summarized, a short summary of old turns, and the most recent turns verbatim. Compare with a typical lossy summary:

In [ ]:
# [offline]
def build_hybrid_history(case_facts: dict, summary: str, turns: list[dict], keep_last: int = 4) -> list[dict]:
    """Facts verbatim + summary of old turns + recent turns verbatim, as a valid messages list."""
    recent = turns[-keep_last:]
    while recent and recent[0]["role"] != "user":          # the array must start with a user turn
        recent = recent[1:]
    context = (f"<case_facts>\n{json.dumps(case_facts, indent=2)}\n</case_facts>\n"
               f"<earlier_conversation_summary>\n{summary}\n</earlier_conversation_summary>")
    first = {"role": "user", "content": [{"type": "text", "text": context}] + as_blocks(recent[0]["content"])}
    return [first] + recent[1:]

long_chat = []
for i in range(1, 13):
    long_chat += [{"role": "user", "content": f"user message {i}"}, {"role": "assistant", "content": f"assistant reply {i}"}]
long_chat.append({"role": "user", "content": "When will my refund arrive?"})   # the new question

facts = {"order_id": "A17", "refund_amount_usd": 42.50, "refund_deadline": "2026-10-10", "customer_tier": "pro"}
lossy_summary = "Customer wants a refund of around 40 dollars for a recent order, sometime next week."
hybrid = build_hybrid_history(facts, "Customer reported a damaged item; refund agreed.", long_chat, keep_last=4)

print("full history:", len(long_chat), "messages | hybrid:", len(hybrid), "messages, roles:", [m["role"] for m in hybrid])
print(hybrid[0]["content"][0]["text"])
print("\nlossy summary alone keeps the exact amount?", "42.50" in lossy_summary)

**Lost-in-the-middle needle test** (small and cheap). Put one fact at the start, middle or end of a filler document and ask for it. The document goes first and the question last.

In [ ]:
# [live] 3 calls of ~4K input tokens, ~ $0.03. Increase FILLER_LINES for a harder test (cost grows linearly).
FILLER_LINES = 150
NEEDLE = "Important: the emergency access code for the Almaty data center is ORCHID-7731."
filler = [f"Note {i}: the weekly infrastructure review covered capacity, patching and backups with no incidents."
          for i in range(FILLER_LINES)]
QUESTION = "Question: what is the emergency access code for the Almaty data center? Reply with the code only."

for position in ("start", "middle", "end"):
    lines = filler.copy()
    lines.insert({"start": 0, "middle": FILLER_LINES // 2, "end": FILLER_LINES}[position], NEEDLE)
    doc = "<document>\n" + "\n".join(lines) + "\n</document>\n\n" + QUESTION
    resp = client.messages.create(**cheap(), messages=[{"role": "user", "content": doc}])
    print(f"{position:<7} found={'ORCHID-7731' in text_of(resp)!s:<5} input_tokens={resp.usage.input_tokens:,}")

> **Observe:** at a few thousand tokens current models usually find the needle everywhere. The effect appears with long inputs full of similar-looking facts, which is why you curate context instead of filling the window.

> **Try it:** change `keep_last` to 3 in `build_hybrid_history`. Why does the function drop the first recent message when it is an `assistant` turn?

### Sampling and non-determinism

At every step the model turns the context into a probability for each possible next token and **samples** one. `temperature` sharpens or flattens that distribution; `top_p` and `top_k` cut off the unlikely tail. Sonnet 5.5, Opus 5.5 and Fable 5.1 reject non-default sampling values (400), and the Python SDK 1.x no longer has these keyword arguments, so on current models you steer with the prompt, `effort` and structured outputs. Either way, identical requests can return different answers, so evals need several runs per example.

[Theory](README.md#sampling-and-non-determinism)

In [ ]:
# [offline] A toy next-token step: what temperature and top-p do to one probability distribution
import math
import random

NEXT_TOKEN_LOGITS = {"billing": 3.0, "technical": 2.2, "account": 1.4, "other": 0.3, "refund": 0.1}


def softmax_with_temperature(logits: dict[str, float], temperature: float) -> dict[str, float]:
    scaled = {t: v / temperature for t, v in logits.items()}
    top = max(scaled.values())
    exp = {t: math.exp(v - top) for t, v in scaled.items()}
    total = sum(exp.values())
    return {t: e / total for t, e in exp.items()}


def top_p_filter(probs: dict[str, float], p: float) -> dict[str, float]:
    """Keep the most likely tokens until their probabilities add up to p, then renormalize."""
    kept, cumulative = {}, 0.0
    for token, prob in sorted(probs.items(), key=lambda kv: kv[1], reverse=True):
        kept[token] = prob
        cumulative += prob
        if cumulative >= p:
            break
    total = sum(kept.values())
    return {t: v / total for t, v in kept.items()}


def bar(probs: dict[str, float]) -> str:
    return "  ".join(f"{t}={v:.2f}" for t, v in sorted(probs.items(), key=lambda kv: -kv[1]))

for temp in (0.3, 1.0, 1.5):
    print(f"temperature {temp:<4} {bar(softmax_with_temperature(NEXT_TOKEN_LOGITS, temp))}")
print(f"top_p 0.9 at T=1.0  {bar(top_p_filter(softmax_with_temperature(NEXT_TOKEN_LOGITS, 1.0), 0.9))}")

rng = random.Random(7)   # seeded only so this demo prints the same thing every time
probs = softmax_with_temperature(NEXT_TOKEN_LOGITS, 1.0)
draws = [rng.choices(list(probs), weights=list(probs.values()))[0] for _ in range(20)]
print("\n20 samples of the SAME step at T=1.0:", " ".join(draws))

> **Observe:** low temperature piles probability onto the top token; high temperature spreads it; top-p drops the tail. Even with a clear favorite, repeated sampling of the *same* step sometimes picks another token, and one different early token changes everything after it. That is non-determinism in miniature.

**Why one eval run is not enough.** Below, a simulated classifier gets most tickets right almost always, but a few tickets are borderline. Compare the spread of accuracy across repeated single-run evals with the spread of a 5-runs-per-ticket pass rate.

In [ ]:
# [offline] Simulated eval: per-ticket probability that the model answers correctly
sim = random.Random(42)
P_CORRECT = [0.98] * 40 + [0.6] * 8 + [0.3] * 2      # 50 tickets: 40 easy, 8 borderline, 2 hard


def one_run_accuracy() -> float:
    return sum(sim.random() < p for p in P_CORRECT) / len(P_CORRECT)


def pass_rates(runs: int) -> list[float]:
    """Fraction of `runs` attempts each ticket got right."""
    return [sum(sim.random() < p for _ in range(runs)) / runs for p in P_CORRECT]

single = [one_run_accuracy() for _ in range(10)]
print("10 single-run evals:     ", " ".join(f"{a:.0%}" for a in single),
      f"| min {min(single):.0%}, max {max(single):.0%}")

multi = [statistics.mean(pass_rates(5)) for _ in range(10)]
print("10 evals, 5 runs/ticket: ", " ".join(f"{a:.0%}" for a in multi),
      f"| min {min(multi):.0%}, max {max(multi):.0%}")

rates = pass_rates(5)
flaky = [i for i, r in enumerate(rates) if 0 < r < 1]
print(f"\nflaky tickets (right on some runs, wrong on others): {len(flaky)} -> {flaky}")

> **Observe:** single runs of the *same* system swing by several points, enough to make a prompt change look better or worse by chance. Averaging several runs per ticket narrows the spread, and the per-ticket pass rate shows *which* tickets are unstable, which is where to look first.

In [ ]:
# [live] Free on current models (a 400 is not billed). Sampling parameters on the default model.
# SDK 1.x has no `temperature` keyword (TypeError), so the only way to send it is extra_body.
try:
    client.messages.create(**cheap(), temperature=0.0, messages=[{"role": "user", "content": "Hi"}])
except TypeError as e:
    print("SDK 1.x:", e)
try:
    resp = client.messages.create(**cheap(), extra_body={"temperature": 0.0},
                                  messages=[{"role": "user", "content": "Reply with one word: hello."}])
    print(f"{MODEL} accepted temperature=0.0 (an older model such as Haiku 4.5):", text_of(resp))
except anthropic.BadRequestError as e:
    print(f"400 on {MODEL} as expected ->", e.message)

In [ ]:
# [live] Opt-in, ~ $0.003: the same request three times. Set RUN_REPEATS = True to run it.
RUN_REPEATS = False
if RUN_REPEATS:
    outputs = []
    for _ in range(3):
        resp = client.messages.create(**cheap(), messages=[{"role": "user", "content": "Suggest a name for a cloud cost dashboard. Reply with the name only."}])
        outputs.append(text_of(resp).strip())
    print(outputs, "| distinct answers:", len(set(outputs)))
else:
    print("Skipped. Set RUN_REPEATS = True to send the same request three times.")

> **Observe:** on the default model `temperature=0.0` is rejected, and with `RUN_REPEATS = True` the same request usually gives more than one distinct answer. Design for that: property checks, several runs per eval example, and structured outputs for the shape.

> **Try it:** in the simulation, change `P_CORRECT` so all 50 tickets are `0.9`. How wide is the single-run spread now? How many runs per ticket do you need before the min-max range of the mean is under 3 points?

## 1.6 Tokens: counting before, measuring after

Before a call, `client.messages.count_tokens(...)` takes the same `system`, `messages` and `tools` and returns an estimate of `input_tokens` (free, model-specific). After a call, `response.usage` reports billed `input_tokens`, `output_tokens` (thinking included) and cache fields. Never use `tiktoken` or chars ÷ 4 for Claude; current models use a newer tokenizer (about 30% more tokens than Haiku 4.5 for the same text).

[Theory](README.md#16-tokens-counting-before-measuring-after)

In [ ]:
# [live] ~ $0.002. Estimate first, then compare with what was billed.
probe = {"system": "You are a concise assistant.",
         "messages": [{"role": "user", "content": "Explain idempotency keys in two sentences."}]}
estimate = client.messages.count_tokens(model=MODEL, **probe)
resp = client.messages.create(**cheap(), **probe)
print(f"count_tokens estimate: {estimate.input_tokens} | usage.input_tokens: {resp.usage.input_tokens} "
      f"| usage.output_tokens: {resp.usage.output_tokens} (thinking included)")

In [ ]:
# [live] Free. Same text, two tokenizers.
sample = ("Customer A17 reported that invoice INV-88 shows two charges of $42.50 on 2026-09-30; "
          "please verify the duplicate, issue a refund, and confirm by email within 2 business days.") * 5
for m in (MODEL, HAIKU):
    try:
        n = client.messages.count_tokens(model=m, messages=[{"role": "user", "content": sample}]).input_tokens
        print(f"{m:<28} {n:>5} tokens")
    except anthropic.NotFoundError:
        print(f"{m:<28} not found (retired? check the deprecations page)")

**Cost from `usage`.** Prices per million tokens from the [pricing page](https://platform.claude.com/docs/en/about-claude/pricing) (October 2026). `usage.input_tokens` is the *uncached* part; cache writes (5-minute TTL) cost 1.25× input, cache reads 0.1× input (0.05× on Opus 5.5, 0.025× on Fable 5.1).

In [ ]:
# [offline]
PRICES = {   # model: (input $/MTok, output $/MTok, cache-read multiplier)
    "claude-fable-5-1": (10.00, 50.00, 0.025),
    "claude-opus-5-5": (4.00, 20.00, 0.05),
    "claude-sonnet-5-5": (2.00, 10.00, 0.10),
    HAIKU: (1.00, 5.00, 0.10),
    "claude-opus-4-8": (5.00, 25.00, 0.10),     # possible fallback targets
    "claude-sonnet-5": (2.00, 10.00, 0.10),
}
CACHE_WRITE_5M = 1.25


def cost_usd(usage, model: str = MODEL) -> float:
    """Dollar cost of one response's usage. Assumes 5-minute cache writes."""
    inp, out, read_mult = PRICES[model]
    cache_write = usage.cache_creation_input_tokens or 0
    cache_read = usage.cache_read_input_tokens or 0
    return (usage.input_tokens * inp + cache_write * inp * CACHE_WRITE_5M
            + cache_read * inp * read_mult + usage.output_tokens * out) / 1e6

demo_usage = SimpleNamespace(input_tokens=1_200, output_tokens=350, cache_creation_input_tokens=0, cache_read_input_tokens=8_000)
for m in ("claude-sonnet-5-5", "claude-opus-5-5", HAIKU):
    print(f"{m:<28} ${cost_usd(demo_usage, m):.5f}")

In [ ]:
# [offline] Statelessness in numbers: each turn re-sends all earlier turns as input
SYSTEM_TOK, USER_TOK, ASSISTANT_TOK = 800, 150, 250
inp, out, _ = PRICES[MODEL]
total_full = total_hybrid = 0.0
print("turn  input tokens (full history)")
for turn in range(1, 51):
    full_input = SYSTEM_TOK + (turn - 1) * (USER_TOK + ASSISTANT_TOK) + USER_TOK
    hybrid_input = SYSTEM_TOK + 300 + min(turn - 1, 4) * (USER_TOK + ASSISTANT_TOK) + USER_TOK  # facts+summary+last 4
    total_full += (full_input * inp + ASSISTANT_TOK * out) / 1e6
    total_hybrid += (hybrid_input * inp + ASSISTANT_TOK * out) / 1e6
    if turn % 5 == 0 or turn == 1:
        print(f"{turn:>4}  {full_input:>6,} {'#' * (full_input // 500)}")
print(f"\n50-turn session cost on {MODEL}: full history ${total_full:.3f} vs hybrid history ${total_hybrid:.3f}")

In [ ]:
# [live] 5 calls, ~ $0.01. Measure the same growth for real.
history: list[dict] = []
for i, q in enumerate(["Name a cloud region in Asia.", "Why might I pick it?", "Name one risk.",
                       "How would I mitigate it?", "Summarize in one line."], 1):
    history.append({"role": "user", "content": q})
    resp = client.messages.create(**cheap(), system="Answer in one short sentence.", messages=history)
    history.append({"role": "assistant", "content": resp.content})
    print(f"turn {i}: input={resp.usage.input_tokens:>4} {'#' * (resp.usage.input_tokens // 10)}  ${cost_usd(resp.usage, resp.model if resp.model in PRICES else MODEL):.5f}")

> **Observe:** input tokens climb every turn even though each answer is short. Prompt caching makes the repeated prefix cheaper; hybrid history or compaction makes it smaller.

> **Try it:** in the simulation, change `keep last 4` to `keep last 10` and see how the hybrid cost moves. What would you have to keep verbatim for a refund conversation?

## 1.7 Model selection basics

Three levers: model tier, effort (`output_config={"effort": ...}`; Haiku 4.5 has none) and architecture (a cheap model for routing, a strong one for synthesis). Choose with a small eval: start capable, step down while the eval passes, and compare **cost per completed task**. Query limits with the Models API instead of hard-coding them.

[Theory](README.md#17-model-selection-basics)

In [ ]:
# [live] Free. Capabilities straight from the Models API
for m in (MODEL, HAIKU):
    try:
        info = client.models.retrieve(m)
        print(f"{info.id:<28} {info.display_name:<20} context={info.max_input_tokens} max_output={info.max_tokens}")
    except anthropic.NotFoundError:
        print(f"{m:<28} not found (retired?)")

In [ ]:
# [offline] A tiny labeled eval set: 12 support tickets, 4 labels
LABELS = ("billing", "technical", "account", "other")
TICKETS = [
    ("My invoice shows two charges for September.", "billing"),
    ("Can I get a refund for the unused months?", "billing"),
    ("Why did my card get declined on renewal?", "billing"),
    ("The API returns 502 errors since this morning.", "technical"),
    ("Deploys hang at the build step after the update.", "technical"),
    ("Our webhook stopped firing for new orders.", "technical"),
    ("How do I add a teammate to my organization?", "account"),
    ("I need to change the email on my login.", "account"),
    ("Please delete my account and all data.", "account"),
    ("Do you have an office in Almaty?", "other"),
    ("Loved the conference talk, thanks!", "other"),
    ("Are you hiring solution architects?", "other"),
]
CLASSIFIER_SYSTEM = (f"Classify the support ticket into exactly one label: {', '.join(LABELS)}. "
                     "Reply with the label only, lowercase, no punctuation.")
print(len(TICKETS), "tickets |", {lbl: sum(t[1] == lbl for t in TICKETS) for lbl in LABELS})

In [ ]:
# [live] 2 configs x 12 tickets = 24 calls, ~ $0.02. Opt into the extras for a fuller comparison (+ ~ $0.10).
CONFIGS = [
    {"name": "Haiku 4.5", "model": HAIKU},
    {"name": "Sonnet 5.5 · low", "model": "claude-sonnet-5-5", "effort": "low"},
]
EXTRA_CONFIGS = [
    {"name": "Sonnet 5.5 · high", "model": "claude-sonnet-5-5", "effort": "high"},
    {"name": "Opus 5.5 · medium", "model": "claude-opus-5-5", "effort": "medium"},
]
RUN_EXTRAS = False


def run_eval(config: dict) -> dict:
    hits, latencies, costs = 0, [], []
    for text, label in TICKETS:
        kwargs = {"model": config["model"], "max_tokens": 2048, "system": CLASSIFIER_SYSTEM,
                  "messages": [{"role": "user", "content": text}]}
        if "effort" in config:
            kwargs["output_config"] = {"effort": config["effort"]}
        t0 = time.perf_counter()
        resp = client.messages.create(**kwargs)
        latencies.append((time.perf_counter() - t0) * 1000)
        hits += text_of(resp).strip().lower().strip(".") == label
        costs.append(cost_usd(resp.usage, config["model"]))
    return {"name": config["name"], "accuracy": hits / len(TICKETS),
            "p50_ms": statistics.median(latencies), "cost_per_1k": 1000 * statistics.mean(costs)}

results = []
for cfg in CONFIGS + (EXTRA_CONFIGS if RUN_EXTRAS else []):
    try:
        results.append(run_eval(cfg))
    except anthropic.NotFoundError:
        print(f"skipping {cfg['name']}: model not found (retired?)")
print(f"{'config':<20} {'accuracy':>8} {'p50 ms':>8} {'$ / 1k tickets':>15}")
for r in results:
    print(f"{r['name']:<20} {r['accuracy']:>8.0%} {r['p50_ms']:>8.0f} {r['cost_per_1k']:>15.3f}")

In [ ]:
# [offline] Pick the cheapest configuration that meets the bar. Illustrative numbers, not measurements.
def pick_config(rows: list[dict], min_accuracy: float, max_p50_ms: float) -> dict | None:
    passing = [r for r in rows if r["accuracy"] >= min_accuracy and r["p50_ms"] <= max_p50_ms]
    return min(passing, key=lambda r: r["cost_per_1k"], default=None)

ILLUSTRATIVE = [
    {"name": "Haiku 4.5", "accuracy": 0.83, "p50_ms": 700, "cost_per_1k": 0.40},
    {"name": "Sonnet 5.5 · low", "accuracy": 0.92, "p50_ms": 1100, "cost_per_1k": 0.75},
    {"name": "Sonnet 5.5 · high", "accuracy": 0.92, "p50_ms": 2600, "cost_per_1k": 2.10},
    {"name": "Opus 5.5 · medium", "accuracy": 0.92, "p50_ms": 2300, "cost_per_1k": 2.60},
]
for bar in (0.80, 0.90, 0.95):
    choice = pick_config(ILLUSTRATIVE, min_accuracy=bar, max_p50_ms=2000)
    print(f"accuracy >= {bar:.0%}, p50 <= 2s -> {choice['name'] if choice else 'nothing passes: step up or improve the prompt'}")

> **Observe:** the decision comes from your eval and your targets, not from "biggest model to be safe". With 12 tickets the accuracy numbers are noisy; a real eval uses 20–100+ representative examples.

> **Try it:** set `RUN_EXTRAS = True`, rerun, then call `pick_config(results, 0.9, 2000)` on your real numbers.

## 1.8 Access paths: SDKs, raw REST, async clients and cloud platforms

The SDK is a typed wrapper over one HTTPS call. This section is fully **offline**: every client gets a *mock transport* (an `httpx2.MockTransport`) that captures the exact HTTP request and returns a canned response, so you can see what each client sends without a key, a cloud account or any cost.

[Theory](README.md#18-access-paths-sdks-raw-rest-async-clients-and-cloud-platforms)

In [ ]:
# [offline] A mock transport: records each outgoing request and answers with a canned Message
import asyncio
import httpx2

CANNED_REPLY = {
    "id": "msg_mock", "type": "message", "role": "assistant", "model": "claude-sonnet-5-5",
    "content": [{"type": "text", "text": "An API gateway is a single entry point that routes, secures and meters API traffic."}],
    "stop_reason": "end_turn", "stop_sequence": None, "stop_details": None,
    "usage": {"input_tokens": 24, "output_tokens": 19},
}
CAPTURED: list[httpx2.Request] = []


def mock_handler(request: httpx2.Request) -> httpx2.Response:
    CAPTURED.append(request)
    return httpx2.Response(200, json=CANNED_REPLY)


def offline(client_cls, **kwargs):
    """Build any SDK client whose HTTP traffic goes to mock_handler instead of the network."""
    return client_cls(**kwargs, http_client=anthropic.DefaultHttpxClient(transport=httpx2.MockTransport(mock_handler)))


def describe(request: httpx2.Request, show_headers=("x-api-key", "authorization", "api-key", "anthropic-version", "content-type")) -> None:
    print(request.method, request.url)
    for name in show_headers:
        if name in request.headers:
            value = request.headers[name]
            print(f"  {name}: {value[:12] + '…' if 'key' in name or name == 'authorization' else value}")
    print("  body:", request.content.decode())

DEMO_REQUEST = {"model": "claude-sonnet-5-5", "max_tokens": 1024,
                "messages": [{"role": "user", "content": "In two sentences, what is an API gateway?"}]}
print("mock transport ready")

In [ ]:
# [offline] 1) What the SDK actually sends for client.messages.create(...)
sdk_client = offline(anthropic.Anthropic, api_key="sk-ant-demo-key", base_url="https://api.anthropic.com")  # explicit, so a local ANTHROPIC_BASE_URL does not change the demo
sdk_reply = sdk_client.messages.create(**DEMO_REQUEST)
sdk_request = CAPTURED[-1]
describe(sdk_request)
print("\nreturned type:", type(sdk_reply).__name__, "| text:", text_of(sdk_reply))
extra = sorted(h for h in sdk_request.headers if h.startswith("x-stainless"))
print("SDK-added telemetry headers:", extra)

In [ ]:
# [offline] 2) The same call as raw REST with httpx2: you set every header and parse the JSON yourself
raw_http = httpx2.Client(transport=httpx2.MockTransport(mock_handler))
raw = raw_http.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-ant-demo-key", "anthropic-version": "2023-06-01", "content-type": "application/json"},
    json=DEMO_REQUEST,
)
raw_request = CAPTURED[-1]
describe(raw_request)
print("\nsame body as the SDK sent?", json.loads(raw_request.content) == json.loads(sdk_request.content))
raw.raise_for_status()                                  # no typed exceptions: you map status codes yourself
message = Message.model_validate(raw.json())            # no typed objects unless you parse into them
print("parsed stop_reason:", message.stop_reason, "| text:", text_of(message))

What the SDK adds on top of the HTTP call: credentials from the environment, `anthropic-version`, timeouts, **automatic retries with backoff**, typed exceptions and typed responses. Watch the retries: the mock below answers `529 overloaded` twice, then succeeds.

In [ ]:
# [offline] 3) Built-in retries: 2 retries by default for connection errors, 408, 409, 429 and 5xx
attempts = {"n": 0}


def flaky_handler(request: httpx2.Request) -> httpx2.Response:
    attempts["n"] += 1
    if attempts["n"] < 3:   # retry-after-ms keeps the demo fast; real servers send their own hint
        return httpx2.Response(529, headers={"retry-after-ms": "20"},
                               json={"type": "error", "error": {"type": "overloaded_error", "message": "Overloaded"}})
    return httpx2.Response(200, json=CANNED_REPLY)


def flaky_client(max_retries: int) -> anthropic.Anthropic:
    return anthropic.Anthropic(api_key="sk-ant-demo-key", max_retries=max_retries,
                               http_client=anthropic.DefaultHttpxClient(transport=httpx2.MockTransport(flaky_handler)))

for retries in (2, 0):
    attempts["n"] = 0
    try:
        flaky_client(retries).messages.create(**DEMO_REQUEST)
        print(f"max_retries={retries}: succeeded after {attempts['n']} HTTP attempts")
    except anthropic.APIStatusError as e:
        print(f"max_retries={retries}: {type(e).__name__} (HTTP {e.status_code}) after {attempts['n']} attempt(s)")

In [ ]:
# [offline] 4) Sync vs async: 5 requests that each take ~0.3 s on the "server"
async def slow_async_handler(request: httpx2.Request) -> httpx2.Response:
    await asyncio.sleep(0.3)
    return httpx2.Response(200, json=CANNED_REPLY)


def slow_sync_handler(request: httpx2.Request) -> httpx2.Response:
    time.sleep(0.3)
    return httpx2.Response(200, json=CANNED_REPLY)

sync_client = anthropic.Anthropic(api_key="sk-ant-demo-key",
                                  http_client=anthropic.DefaultHttpxClient(transport=httpx2.MockTransport(slow_sync_handler)))
t0 = time.perf_counter()
for _ in range(5):
    sync_client.messages.create(**DEMO_REQUEST)
print(f"sync,  one after another: {time.perf_counter() - t0:.2f} s")

async_client = anthropic.AsyncAnthropic(api_key="sk-ant-demo-key",
                                        http_client=anthropic.DefaultAsyncHttpxClient(transport=httpx2.MockTransport(slow_async_handler)))
limit = asyncio.Semaphore(3)   # stay under your rate limit: at most 3 in flight


async def one_call():
    async with limit:
        return await async_client.messages.create(**DEMO_REQUEST)

t0 = time.perf_counter()
replies = await asyncio.gather(*(one_call() for _ in range(5)))   # top-level await works in Jupyter
print(f"async, 3 in flight:       {time.perf_counter() - t0:.2f} s for {len(replies)} replies")

> **Observe:** the request body is identical whether the SDK or raw HTTP sends it; the SDK adds the plumbing. With `max_retries=0` the first 529 surfaces as `OverloadedError`. The async client finishes the same five calls in about two "server" round trips instead of five, without changing a single request field.

**Cloud platforms.** Each platform has its own client class; the request body stays the Messages API. Watch where the model ID and the credentials go. (All four clients below get explicit dummy credentials, so nothing is read from your machine and nothing leaves it.)

In [ ]:
# [offline] 5) The same request through four platform clients
PLATFORM_CLIENTS = [
    ("Claude API", anthropic.Anthropic, {"api_key": "sk-ant-demo-key", "base_url": "https://api.anthropic.com"}, "claude-sonnet-5-5"),
    ("Claude Platform on AWS", anthropic.AnthropicAWS,
     {"aws_region": "us-east-1", "workspace_id": "wrkspc_demo", "api_key": "demo-aws-key"}, "claude-sonnet-5-5"),
    ("Amazon Bedrock", anthropic.AnthropicBedrockMantle,
     {"aws_region": "us-east-1", "api_key": "demo-bedrock-bearer-token"}, "anthropic.claude-sonnet-5-5"),
    ("Google Cloud (Vertex AI)", anthropic.AnthropicVertex,
     {"project_id": "demo-project", "region": "global", "access_token": "demo-gcp-token"}, "claude-sonnet-5-5"),
    ("Microsoft Foundry", anthropic.AnthropicFoundry,
     {"api_key": "demo-azure-key", "resource": "example-resource"}, "claude-sonnet-5-5"),   # = deployment name
]
for label, cls, kwargs, model_id in PLATFORM_CLIENTS:
    reply = offline(cls, **kwargs).messages.create(**(DEMO_REQUEST | {"model": model_id}))
    print(f"--- {label} ({cls.__name__})")
    describe(CAPTURED[-1])
    print()

> **Observe:** Bedrock keeps the Messages API path but adds the `anthropic.` prefix to the model ID. Google Cloud moves the model **into the URL** (`…/models/claude-sonnet-5-5:rawPredict`) and puts `anthropic_version` in the body. Foundry and Claude Platform on AWS keep bare model IDs (on Foundry the value is your deployment name). Your `messages.create(...)` call barely changes; configuration does.

Keep those differences in one place: a per-platform model map plus a list of features each platform lacks (October 2026; check the Features overview page before relying on it).

In [ ]:
# [offline] 6) A per-platform request planner you could keep in configuration
MISSING_FEATURES = {
    "anthropic": set(),
    "bedrock": {"structured_outputs", "message_batches", "models_api", "server_side_fallback",
                "web_search", "web_fetch", "code_execution", "files_api"},
    "vertex": {"message_batches", "models_api", "server_side_fallback", "web_fetch", "code_execution", "files_api"},
    "foundry": {"message_batches", "models_api", "server_side_fallback"},   # Hosted-on-Azure deployments lack more
}


def platform_model_id(platform: str, model: str, deployment: str | None = None) -> str:
    dated = re.fullmatch(r"(.+)-(\d{8})", model)            # e.g. claude-haiku-4-5-20251001
    if platform == "anthropic":
        return model
    if platform == "bedrock":
        return "anthropic." + (dated.group(1) if dated else model)
    if platform == "vertex":
        return f"{dated.group(1)}@{dated.group(2)}" if dated else model
    if platform == "foundry":
        return deployment or (dated.group(1) if dated else model)
    raise ValueError(f"unknown platform {platform!r}")

NEEDS = {"prompt_caching", "structured_outputs", "message_batches"}
for platform in MISSING_FEATURES:
    print(f"{platform:<10} model={platform_model_id(platform, HAIKU):<30} missing for this app: {sorted(NEEDS & MISSING_FEATURES[platform]) or '-'}")

> **Observe:** the same app needs a different model string on every platform, and two of the three features it relies on disappear on Bedrock. That is an architecture decision (validation and retry instead of structured outputs, real-time calls instead of batches), not a code tweak.

> **Try it:** add `"web_search"` to `NEEDS`. Which platforms still work? Then build the Bedrock client with `max_retries=0` and the `flaky_handler` transport: what does a Bedrock 529 look like to your code?

## Mini-project: a multi-turn chat helper that never mishandles a `stop_reason`

`ChatSession` combines the whole chapter:

- keeps the history and resends it every turn (1.2), with the system prompt at the top level (1.4)
- handles **all seven** stop reasons (1.3):
  - `end_turn` / `stop_sequence`: return the text
  - `tool_use`: run every requested tool and send all results in one `user` turn
  - `pause_turn`: send the paused assistant turn back unchanged
  - `max_tokens`: retry with a bigger budget when the turn is unusable (no text, or a half-written tool call), otherwise ask the model to continue
  - `model_context_window_exceeded`: report truncation and roll back (1.5)
  - `refusal`: switch to a fallback model and stay there, or roll back cleanly
- records `usage` and cost for every call (1.6) and works with any model you pass in (1.7)

It takes the client as a parameter, so we can test every branch **offline** with a scripted fake client before running it live.

In [ ]:
class TruncatedError(RuntimeError):
    pass


@dataclass
class Reply:
    text: str
    stop_reason: str
    model: str
    truncated: bool = False
    refused: bool = False
    note: str = ""


@dataclass
class UsageLedger:
    calls: int = 0
    input_tokens: int = 0
    output_tokens: int = 0
    cache_read: int = 0
    cache_write: int = 0
    cost: float = 0.0
    unpriced_calls: int = 0

    def add(self, resp) -> None:
        u = resp.usage
        self.calls += 1
        self.input_tokens += u.input_tokens
        self.output_tokens += u.output_tokens
        self.cache_read += u.cache_read_input_tokens or 0
        self.cache_write += u.cache_creation_input_tokens or 0
        if resp.model in PRICES:
            self.cost += cost_usd(u, resp.model)
        else:
            self.unpriced_calls += 1


class ChatSession:
    def __init__(self, system: str, *, client=client, model: str = MODEL, effort: str | None = "low",
                 max_tokens: int = 2048, max_tokens_cap: int = 16_000, tools: list[dict] | None = None,
                 tool_impls: dict[str, Callable[..., Any]] | None = None, fallback_model: str | None = None,
                 stop_sequences: list[str] | None = None, max_steps: int = 8):
        self.system, self.client, self.model, self.effort = system, client, model, effort
        self.max_tokens, self.max_tokens_cap = max_tokens, max_tokens_cap
        self.tools, self.tool_impls = tools or [], tool_impls or {}
        self.fallback_model, self.stop_sequences, self.max_steps = fallback_model, stop_sequences, max_steps
        self.history: list[dict] = []
        self.ledger = UsageLedger()

    # -- one API call ------------------------------------------------------
    def _create(self):
        kwargs: dict[str, Any] = {"model": self.model, "max_tokens": self.max_tokens,
                                  "system": self.system, "messages": self.history}
        if self.effort and supports_effort(self.model):
            kwargs["output_config"] = {"effort": self.effort}
        if self.tools:
            kwargs["tools"] = self.tools
        if self.stop_sequences:
            kwargs["stop_sequences"] = self.stop_sequences
        resp = self.client.messages.create(**kwargs)
        self.ledger.add(resp)
        return resp

    def _run_tools(self, resp) -> list[dict]:
        results = []
        for block in resp.content:
            if block.type != "tool_use":
                continue
            fn = self.tool_impls.get(block.name)
            try:
                if fn is None:
                    raise KeyError(f"unknown tool {block.name!r}")
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(fn(**block.input))})
            except Exception as e:  # report tool failures to the model instead of crashing the turn
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": f"{type(e).__name__}: {e}", "is_error": True})
        return results

    # -- one user turn -----------------------------------------------------
    def send(self, user_text: str) -> Reply:
        checkpoint = len(self.history)
        self.history.append({"role": "user", "content": user_text})
        pieces: list[str] = []

        def rollback() -> None:
            del self.history[checkpoint:]

        for _ in range(self.max_steps):
            resp = self._create()
            match resp.stop_reason:
                case "end_turn" | "stop_sequence":
                    self.history.append({"role": "assistant", "content": resp.content})
                    pieces.append(text_of(resp))
                    note = f"stopped on {resp.stop_sequence!r}" if resp.stop_sequence else ""
                    return Reply("".join(pieces), resp.stop_reason, resp.model, note=note)
                case "tool_use":
                    self.history.append({"role": "assistant", "content": resp.content})
                    self.history.append({"role": "user", "content": self._run_tools(resp)})
                case "pause_turn":
                    self.history.append({"role": "assistant", "content": resp.content})   # unchanged, no new user turn
                case "max_tokens":
                    unusable = not text_of(resp) or any(b.type == "tool_use" for b in resp.content)
                    if unusable:   # budget spent on thinking, or a half-written tool call: never execute it
                        if self.max_tokens >= self.max_tokens_cap:
                            rollback()
                            return Reply("".join(pieces), "max_tokens", resp.model, truncated=True,
                                         note="max_tokens cap reached; history rolled back")
                        self.max_tokens = min(self.max_tokens * 2, self.max_tokens_cap)
                        continue
                    pieces.append(text_of(resp))
                    self.history.append({"role": "assistant", "content": resp.content})
                    self.history.append({"role": "user", "content": "Continue exactly where you stopped. Do not repeat anything."})
                case "model_context_window_exceeded":
                    rollback()
                    return Reply("".join(pieces) + text_of(resp), resp.stop_reason, resp.model, truncated=True,
                                 note="context window full: compact or trim history, then resend; history rolled back")
                case "refusal":
                    if self.fallback_model and self.model != self.fallback_model:
                        self.model = self.fallback_model           # sticky: later turns stay on the fallback
                        continue                                   # partial output (if any) is discarded
                    rollback()
                    category = resp.stop_details.category if resp.stop_details else None
                    return Reply("", "refusal", resp.model, refused=True, note=f"category={category}")
                case other:
                    rollback()
                    raise RuntimeError(f"Unhandled stop_reason {other!r}")
        rollback()
        raise RuntimeError(f"No final answer after {self.max_steps} model calls")

    def report(self) -> None:
        L = self.ledger
        unpriced = f" (+{L.unpriced_calls} unpriced)" if L.unpriced_calls else ""
        print(f"calls={L.calls} input={L.input_tokens:,} output={L.output_tokens:,} "
              f"cache_read={L.cache_read:,} cache_write={L.cache_write:,} cost=${L.cost:.4f}{unpriced}")
        print(f"history={len(self.history)} messages | current model={self.model} | max_tokens={self.max_tokens}")

**Offline test bench.** `ScriptedClient` stands in for `anthropic.Anthropic()`: it returns queued responses and records a snapshot of every request, so each branch can be checked without spending anything.

In [ ]:
# [offline]
class ScriptedClient:
    def __init__(self, responses: list[Message]):
        self.responses = list(responses)
        self.requests: list[dict] = []
        self.messages = SimpleNamespace(create=self._create)

    def _create(self, **kwargs):
        self.requests.append({**kwargs, "messages": list(kwargs["messages"])})   # snapshot
        return self.responses.pop(0)


def scripted(*responses, **session_kwargs) -> tuple[ChatSession, ScriptedClient]:
    fake = ScriptedClient(list(responses))
    return ChatSession("You are a test bot.", client=fake, **session_kwargs), fake

ORDER_TOOL = [{"name": "get_order", "description": "Look up an order by ID.",
               "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}]
final = fake_message([{"type": "text", "text": "Done."}])

# tool_use -> end_turn
s, f = scripted(MOCKS["tool_use"], final, tools=ORDER_TOOL, tool_impls={"get_order": lambda order_id: f"{order_id}: shipped"})
r = s.send("Where is A17?")
last = f.requests[1]["messages"][-1]
assert r.text == "Done." and last["role"] == "user" and last["content"][0]["type"] == "tool_result"
assert last["content"][0]["content"] == "A17: shipped"
print("✓ tool_use: tool ran, result sent in a user turn")

# pause_turn -> end_turn: second request ends with the paused assistant turn, no extra user message
s, f = scripted(MOCKS["pause_turn"], final)
s.send("Search the news")
assert [m["role"] for m in f.requests[1]["messages"]] == ["user", "assistant"]
print("✓ pause_turn: resent unchanged")

# max_tokens with only a thinking block -> retry with a doubled budget
thinking_only = fake_message([{"type": "thinking", "thinking": "", "signature": "s"}], "max_tokens")
s, f = scripted(thinking_only, final)
s.send("Hard question")
assert f.requests[1]["max_tokens"] == 2 * f.requests[0]["max_tokens"]
print("✓ max_tokens (no text): retried with", f.requests[1]["max_tokens"], "tokens")

# max_tokens with a half-written tool call -> NOT executed
half_tool = fake_message([{"type": "tool_use", "id": "toolu_x", "name": "get_order", "input": {}}], "max_tokens")
calls = []
s, f = scripted(half_tool, final, tools=ORDER_TOOL, tool_impls={"get_order": lambda **kw: calls.append(kw)})
s.send("Where is A17?")
assert calls == []
print("✓ max_tokens (partial tool_use): tool not executed")

# max_tokens with partial text -> continue and stitch
s, f = scripted(fake_message([{"type": "text", "text": "Part one, "}], "max_tokens"),
                fake_message([{"type": "text", "text": "part two."}]))
r = s.send("Write two parts")
assert r.text == "Part one, part two." and f.requests[1]["messages"][-1]["role"] == "user"
print("✓ max_tokens (partial text): continued ->", repr(r.text))

# refusal with a fallback model -> retried on the fallback, which stays sticky
s, f = scripted(MOCKS["refusal"], fake_message(model="claude-opus-4-8"), fallback_model="claude-opus-4-8")
r = s.send("Review this config")
assert f.requests[1]["model"] == "claude-opus-4-8" and s.model == "claude-opus-4-8" and not r.refused
print("✓ refusal + fallback: answered by", r.model)

# refusal without a fallback -> clean rollback, history stays valid
s, f = scripted(MOCKS["refusal"])
r = s.send("Review this config")
assert r.refused and s.history == [] and "cyber" in r.note
print("✓ refusal: rolled back,", r.note)

# model_context_window_exceeded -> truncated, rolled back
s, f = scripted(MOCKS["model_context_window_exceeded"])
r = s.send("Summarize all filings")
assert r.truncated and s.history == []
print("✓ model_context_window_exceeded: truncated, rolled back")

# stop_sequence -> returned with the matched sequence
s, f = scripted(MOCKS["stop_sequence"], stop_sequences=["7"])
r = s.send("Count")
assert r.stop_reason == "stop_sequence" and "'7'" in r.note
print("✓ stop_sequence:", r.note)
s.report()

**Live run.** A short support conversation with one real tool, then the usage and cost report.

In [ ]:
# [live] ~ 4–6 calls, ~ $0.02
ORDERS = {"A17": "shipped 2026-10-01 via DHL, ETA 2026-10-06", "B42": "processing, ships within 2 business days"}

session = ChatSession(
    "You are Acme's order-support assistant. Use get_order for order questions. Answer in at most 40 words.",
    tools=ORDER_TOOL,
    tool_impls={"get_order": lambda order_id: ORDERS.get(order_id.upper(), "no such order")},
)
for question in ["Hi, where is my order A17?", "And B42?", "Which of the two arrives first?"]:
    reply = session.send(question)
    print(f"> {question}\n{reply.text}  [{reply.stop_reason}{' · ' + reply.note if reply.note else ''}]\n")
session.report()

> **Observe:** turn 3 needs no tool call because the earlier tool results are in the history the session resends. The report shows how many model calls a "3-question" chat really took, and what they cost.

> **Try it:** construct the session with `max_tokens=16`; watch the budget double until a usable answer appears. Then add `stop_sequences=["ETA"]` and see the `stop_sequence` branch fire for real.

## Recap

- One endpoint, `POST /v1/messages`: `model`, `max_tokens`, `messages` are required; read `content` **by block type**; errors are typed exceptions, not stop reasons.
- `user`/`assistant` alternate; tool results go in a `user` turn; the API is stateless, so the app resends the full history; mid-conversation `system` messages are model-gated; prefill is gone on current models.
- Seven `stop_reason` values, each with its own action. `stop_details` exists only for `refusal`; `max_tokens` and `model_context_window_exceeded` need opposite fixes.
- The system prompt holds persistent rules; scope tool-related instructions to avoid unintended tool use; fight drift with reminders before validation.
- Context is a budget: trim tool results, keep a case-facts block, put key facts at the edges.
- Outputs vary between identical requests (sampling, adaptive thinking); current models reject non-default `temperature`. Evaluate with several runs per example and property checks.
- Count with `count_tokens` (free, model-specific) before; read `usage` after; compute cost per completed task.
- Pick the cheapest model and effort that pass your eval.
- The SDK is a typed wrapper over one HTTPS call (plus retries, timeouts, typed errors); `AsyncAnthropic` keeps many calls in flight; Bedrock, Google Cloud and Foundry use their own client classes, model IDs and feature subsets.

**Next:** [homework](02_homework.ipynb)